In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
from flask import Flask, request, jsonify
import threading, time
from google.colab import output

# 1. Train the model fresh (fixed)
print("Downloading and training model...")
url = "https://raw.githubusercontent.com/pandas-dev/pandas/master/doc/data/titanic.csv"
df = pd.read_csv(url)

df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, errors='ignore')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna('S')

le_sex = LabelEncoder()
le_embarked = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])

X = df.drop('Survived', axis=1)
y = df['Survived']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

joblib.dump(model, 'titanic_model.pkl')
joblib.dump(le_sex, 'le_sex.pkl')
joblib.dump(le_embarked, 'le_embarked.pkl')
print("✅ Model trained and saved!")

# 2. Start Flask API
app = Flask(__name__)
model = joblib.load('titanic_model.pkl')
le_sex = joblib.load('le_sex.pkl')
le_embarked = joblib.load('le_embarked.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array([[
            int(data['Pclass']),
            le_sex.transform([str(data['Sex']).lower()])[0],
            float(data['Age']),
            int(data['SibSp']),
            int(data['Parch']),
            float(data['Fare']),
            le_embarked.transform([str(data['Embarked']).upper()])[0]
        ]])
        survived = int(model.predict(features)[0])
        prob = round(float(model.predict_proba(features)[0][1]), 4)
        return jsonify({"Survived": survived, "Survival_Probability": prob})
    except Exception as e:
        return jsonify({"error": str(e)}), 400

def run_server():
    app.run(host='0.0.0.0', port=5000, debug=False)

print("Starting Flask server...")
threading.Thread(target=run_server, daemon=True).start()
time.sleep(6)

print("\n🎉 API is ready!")
output.serve_kernel_port_as_window(5000)
print("→ Click the link above to open")
print("→ Or test with the next cell")

✅ Model trained and saved!
Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



🎉 API is ready!
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

→ Click the link above to open
→ Or test with the next cell


In [ ]:
import requests
payload = {"Pclass":3, "Sex":"male", "Age":25, "SibSp":0, "Parch":0, "Fare":7.25, "Embarked":"S"}
r = requests.post("http://127.0.0.1:5000/predict", json=payload, timeout=10)
print(r.json())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [05/Apr/2026 10:33:14] "POST /predict HTTP/1.1" 200 -


{'Survival_Probability': 0.03, 'Survived': 0}
